In [1]:
!pip install -q datasets tensorflow scikit-learn

In [2]:
!pip install -q datasets tensorflow scikit-learn
# Data from WikiText
from datasets import load_dataset
import os

TARGET_WORDS = 10_000_000
DATA_FILE = "wikitext_10M.txt"

if not os.path.exists(DATA_FILE):

    print("Downloading WikiText-103...")

    dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-103-raw-v1",
        split="train"
    )

    words = []
    count = 0

    for row in dataset:
        text = row["text"].strip()

        if not text:
            continue

        tokens = text.split()

        words.extend(tokens)
        count += len(tokens)

        if count >= TARGET_WORDS:
            break

    corpus = " ".join(words[:TARGET_WORDS])

    with open(DATA_FILE, "w", encoding="utf-8") as f:
        f.write(corpus)

    print(f"Saved {TARGET_WORDS:,} words.")

else:
    print("Dataset already exists.")

#Load Data
with open(DATA_FILE, "r", encoding="utf-8") as f:
    text = f.read()

print("Characters:", len(text))
print("Words:", len(text.split()))

text[:500]    

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Saved 10,000,000 words.
Characters: 52825461
Words: 10000000


'= Valkyria Chronicles III = Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs paral'

In [14]:
#Load Data
with open(DATA_FILE, "r", encoding="utf-8") as f:
    text = f.read()

print("Characters:", len(text))
print("Words:", len(text.split()))

text[:500]

Characters: 52825461
Words: 10000000


'= Valkyria Chronicles III = Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs paral'

In [15]:
#Conv. text into no.
from tensorflow.keras.preprocessing.text import Tokenizer

VOCAB_SIZE = 50000

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts([text])

total_words = min(
    VOCAB_SIZE,
    len(tokenizer.word_index) + 1
)

print("Vocabulary Size:", total_words)

Vocabulary Size: 50000


In [5]:
import numpy as np

words = text.split()

SEQ_LEN = 50

sequences = []

for i in range(SEQ_LEN, len(words)):

    seq = words[i-SEQ_LEN:i+1]

    token_list = tokenizer.texts_to_sequences(
        [" ".join(seq)]
    )[0]

    if len(token_list) == SEQ_LEN + 1:
        sequences.append(token_list)

sequences = np.array(sequences)

print(sequences.shape)

(2725, 51)


In [6]:
#init x,y

X = sequences[:, :-1]
y = sequences[:, -1]

print(X.shape)
print(y.shape)


(2725, 50)
(2725,)


In [7]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.layers import BatchNormalization, Dropout, Embedding, Bidirectional, SimpleRNN, LSTM, GRU, Dense, Input
import numpy as np

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

EMBED_DIM = 256

model = Sequential([
    Embedding(
        input_dim=total_words,
        output_dim=EMBED_DIM,
        input_shape=(SEQ_LEN,)
    ),

    Bidirectional(
        LSTM(
            256,
            return_sequences=True
        )
    ),

    Dropout(0.3),

    Bidirectional(
        LSTM(256)
    ),

    Dense(
        512,
        activation='relu'
    ),

    Dropout(0.3),

    Dense(
        total_words,
        activation='softmax'
    )
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 256)        │    12,800,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 50, 512)        │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 512)            │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50000)          │    25,650,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,338,192 (157.69 MB)

 Trainable params: 41,338,192 (157.69 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [9]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

optimizer = AdamW(
    learning_rate=1e-3,
    weight_decay=1e-5
)  

checkpoint = ModelCheckpoint(
    "best_next_word.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
) 

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1
)   

In [10]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [11]:
history = model.fit(
    X, y,
    epochs=15,

    batch_size=128,
    validation_split=0.2,
    callbacks = 
    [early_stop,
    reduce_lr,
    checkpoint]
)

Epoch 1/15
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.0461 - loss: 10.4157
Epoch 1: val_loss improved from None to 8.51526, saving model to best_next_word.keras

Epoch 1: finished saving model to best_next_word.keras
18/18 ━━━━━━━━━━━━━━━━━━━━ 19s 384ms/step - accuracy: 0.0560 - loss: 9.7090 - val_accuracy: 0.0606 - val_loss: 8.5153 - learning_rate: 0.0010
Epoch 2/15
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.0267 - loss: 7.2769
Epoch 2: val_loss did not improve from 8.51526
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 92ms/step - accuracy: 0.0335 - loss: 7.1849 - val_accuracy: 0.0606 - val_loss: 8.8183 - learning_rate: 0.0010
Epoch 3/15
17/18 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.0618 - loss: 6.5978
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 3: val_loss did not improve from 8.51526
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 99ms/step - accuracy: 0.0674 - loss: 6.5939 - val_accuracy: 0.0606 - val_loss: 9.1434 - learning_rate: 0.0010
Epoch 4/1

In [12]:
def generate_text(
    model,
    tokenizer,
    seed_text,
    seq_len,
    num_words=20
):

    current_text = seed_text

    for _ in range(num_words):

        token_list = tokenizer.texts_to_sequences(
            [current_text]
        )[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=seq_len,
            padding='pre'
        )

        predicted_probs = model.predict(
            token_list,
            verbose=0
        )

        predicted_index = np.argmax(
            predicted_probs
        )

        predicted_word = ""

        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                predicted_word = word
                break

        current_text += " " + predicted_word

    return current_text

In [13]:
seed = "machine learning"

generated = generate_text(
    model,
    tokenizer,
    seed,
    SEQ_LEN,
    num_words=30
)

print(generated)

machine learning the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the
